In [1]:
import json

In [6]:
clusters = {}
with open('/scratch/fliu/data/cliff/ani_mmseqs/output/mmseqs_cluster.tsv') as fh:
    l = fh.readline()
    while(l):
        if l:
            a, b = l.split()
            if a not in clusters:
                clusters[a] = set()
            clusters[a].add(b)
        l = fh.readline()

In [126]:

genome_sets = {}
with open('/scratch/fliu/data/cliff/genome_clusters_84.json', 'r') as fh:
    genome_sets = json.load(fh)

In [127]:
for k in genome_sets:
    if len(genome_sets[k]) > 5:
        print(k)

Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_concoct_out.60.contigs__.RAST
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.28.contigs__.RAST
Salt_Pond_MetaGSF2_A_D2_MG_DASTool_bins_concoct_out.13.contigs__.RAST
Salt_Pond_MetaG_R2_B_D1_MG_DASTool_bins_metabat.22.contigs__.RAST
Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.metabat.26.contigs__.RAST
Salt_Pond_MetaG_R2_restored_C_black_MG_DASTool_bins_concoct_out.17.contigs__.RAST
Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_concoct_out.29.contigs__.RAST
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.19.contigs__.RAST
Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_metabat.14.contigs__.RAST
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.38.contigs__.RAST
Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_metabat.27.contigs__.RAST
Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.11.contigs__.RAST
Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_concoct_out.65.contigs__.RAST
Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_metabat.12.contigs__.RAST
Salt_Pond_MetaGSF2_A_D2_MG_DASTool_bins

In [121]:
genome_sets['Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.19.contigs__.RAST']

['612a026bf84a8de6d03f46adcb37524f8836f6cfb75f09874b851d75e2930321',
 '51d4eec6d0d46802bdb2e828df628fb6d3a520ca4e180fa6166e7871d35ec223',
 '04a4faf0370450b9e74ad783f7e6263dcaf53e9a7542e75447b66955f79dad9f',
 '19ec9d6ade7e21b16fcf566e2b4580573dafd356cec9155ad202d902ab235236',
 '73fe326554fa4f52d2d38bb03dd9576ab9f44859b3224457801096dc6141cd37',
 '6a4f6807c187757294bc161f2b85951f24380099d6e4bdf7b048d57961db69eb',
 '980658fbf3be3b107bd74c6f4fe1bda2b940bd15e57d47edd66dab860fa64473',
 '9c8c2ce96f0c16475f23bcb9252518679ab041df273150c12724b2e0835240ff',
 'c300f97c1d9a5205eb607c2910a5ba6912616b5996e354f480d803579947d2ae',
 '27899bd956c9c61a03845e24e21340432f9966b1feac897ac214a67b9620d701']

In [128]:
members = set()

for k in genome_sets:
    if len(genome_sets[k]) > 1:
        members |= set(genome_sets[k])
len(members)

386

In [13]:
feature_to_genome = {}
with open('/scratch/fliu/data/cliff/metadata_ani_g.json', 'r') as fh:
    feature_to_genome = json.load(fh)

In [18]:
def sub_clusters(clusters, f_list):
    res = {}
    for a, b in clusters.items():
        valid = set()
        for _b in b:
            g = feature_to_genome.get(_b)
            if g in f_list:
                valid.add(_b)
        if len(valid) > 0:
            l = list(valid)
            res[l[0]] = l
    return res
f = list(genome_sets['Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.19.contigs__.RAST'])
s_clusters = sub_clusters(clusters, f)


In [14]:
for k in feature_to_genome:
    print(k, feature_to_genome[k])
    break

NZ_RWVB01000001.1_1 8d6a7c6becfe66e81095503c84c242c2e40201ec0dfa33f71d97f6fc8e618679


In [ ]:
core_data_comp = {}
for mag in genome_sets:
    gset = genome_sets[mag]
    if len(gset) > 1:
        print(gset)
        f = list(genome_sets[mag])
        s_clusters = sub_clusters(clusters, f)
        #genome2gene_clusters, recs, gene_clusters2rep = load_mmseq_results4(s_clusters, feature_to_genome, all_genomes, 'c')
        filter_features = set()
        for a, b in s_clusters.items():
            filter_features |= set(b)
        recs = get_recs(filter_features, all_cc)
        genome2gene_clusters2, recs2, gene_clusters2rep2 = load_mmseq_results4(recs, feature_to_genome, f, 'cc')
        tt = {
            'genome2gene_clusterss' : genome2gene_clusters2, 
            'aa2gene_clusters' : recs2, 
            'gene_clusters2rep' : gene_clusters2rep2}
        fast_motu = FastMOTU(tt)
        fast_motu.run()
        print(len(fast_motu), fast_motu.mean_comp)
        m_out = get_stats(fast_motu, 'hi')
        
        cluster_to_features = {}
        for f_id, c_id in fast_motu.aa2gene_clusters.items():
            if c_id not in cluster_to_features:
                cluster_to_features[c_id] = set()
            cluster_to_features[c_id].add(f_id)
        feature_id_to_genome = {}
        for feature_ids in cluster_to_features.values():
            for feature_id in feature_ids:
                genome_id = feature_to_genome.get(feature_id)
                if genome_id:
                    feature_id_to_genome[feature_id] = genome_id
                else:
                    print('!!')
        core_data =  {
            'core': m_out['hi']['core'],
            'cluster_to_features': cluster_to_features,
            'feature_id_to_genome': feature_id_to_genome
        }
        core_data_comp[mag] = core_data

['Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_concoct_out.51.contigs.fa_assembly.RAST', 'ede5a1e394f39e509742d01574f8121b1d2c2470ec68f97d86877bb503cc823f', 'cd3e1d1fcbae456da76bbfc6f3f15e538adcdbdd11c0bcc013bf8667ca9fe042', '2a00b09cf8fa0a52c434fbde82e8a9a1d4e967aa340f8d7b93a5f90bc4b5cc81', 'Salt_Pond_MetaG_R2_B_H2O_MG_DASTool_bins_concoct_out.4.contigs.fa_assembly.RAST', 'e393c653dde4c5f7feb8d4465dbb2796796fbad404f523a279490d0d8a1e1fc8']


iteration 1 :  0 sum_abs_LLHR: 51736.06409326803
iteration 2 :  0 sum_abs_LLHR: 40517.85408043911

Your default-run for 6 genomes (with mean initial completeness 99.90) resulted
in a core of 0 traits with a total sum of loglikelihood-ratios 40517.85 and a corrected 
mean completness of 0.01, resulting to a estimated mean traits per genome count of 19350000.00



6 100.0
['2fe1ce062950fbb243e4c78ae6522f149c3b60abfb1598d23a744790a8f03da5', '5546c56f78db472d74ba9ab9f811c65fec5a14bc8cb621a8a2e2ec7c03b18ab4', '29f819d0f72d64e494b3f081cd0e07dbe0aed7ca30f42e5bd026445135707a8b', '69eb186e7009b2efdc5e604dcc642df656eff32be3fa63f27b6b8698c8e7f9ae', 'ce62c9bde5a8e6305316812bcdfcd0df982606abab397e7bb939267a7076d564', '0a8dabb32e5ab8e8b2ea0022ea4425a9c6037fbfe5fd524394abc5afacd38a76', '05b6597344692271c507aed14f685f1d4c3b3692ecf9711f543a31d950d87564', '2cc0708cd5c9a826e3100481f015bcd679a16d0389c53ef27b0f5c9288ef9d85', 'f89bc436f4eee23b4737d8088ce173f18dc69e28440c80515e579485f88f70ad', '1b0a21ed41e6a0eb9cb3eb384da59783c26d83ef8d1cec971c8e2cc929fa1c8f']


iteration 1 :  663 sum_abs_LLHR: 42186.402274211556
iteration 2 :  663 sum_abs_LLHR: 42186.402274211556

Your default-run for 10 genomes (with mean initial completeness 99.90) resulted
in a core of 663 traits with a total sum of loglikelihood-ratios 42186.40 and a corrected 
mean completness of 99.90, resulting to a estimated mean traits per genome count of 1518.62



10 100.0
['Salt_Pond_MetaGSF2_C_H2O_MG_DASTool_bins_metabat.17.contigs.fa_assembly.RAST', '0cdf9c2a300fbc7087b2a1c8038ece05b179c1ba59200c5bdf0bcaf99396cc16', 'Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.15.contigs.fa_assembly.RAST', '8f852dce27c8359e4370a08b4b8cda00bc74e4f35ddcdadcee372341b48f971b']


...

4 100.0
['d809e7075c2b547e9e0414f5482b2042747badebe1cfa8a3c953b3d25ab5753a', 'Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_metabat.13.contigs.fa_assembly.RAST', '11f1f8d0534d340ae171aa494a2c481bf4b71cc3de588ac122eeb8fa878ab464', '1836438502ddccabf165f81210770f1722442376bb2fb75956d5b726f1efc3cb', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.11.contigs.fa_assembly.RAST']


iteration 1 :  0 sum_abs_LLHR: 36845.06420034055
iteration 2 :  0 sum_abs_LLHR: 30191.85201686937

Your default-run for 5 genomes (with mean initial completeness 99.90) resulted
in a core of 0 traits with a total sum of loglikelihood-ratios 30191.85 and a corrected 
mean completness of 0.01, resulting to a estimated mean traits per genome count of 17570000.00



5 100.0
['0512d9aa6b787efbc085bbcce83a1ed36e35fcf7fa895fce6526b56d50db187a', '39aaeffe52bf0b1fc2cb65c8d51be6fcefe0bcbfe1533fb9bccda28a636978a4', 'Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_metabat.15.contigs.fa_assembly.RAST']


iteration 1 :  0 sum_abs_LLHR: 11675.818964570317
iteration 2 :  0 sum_abs_LLHR: 18363.80595626119

Your default-run for 3 genomes (with mean initial completeness 99.90) resulted
in a core of 0 traits with a total sum of loglikelihood-ratios 18363.81 and a corrected 
mean completness of 0.01, resulting to a estimated mean traits per genome count of 16960000.00



3 100.0
['d809e7075c2b547e9e0414f5482b2042747badebe1cfa8a3c953b3d25ab5753a', 'Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_metabat.13.contigs.fa_assembly.RAST', '11f1f8d0534d340ae171aa494a2c481bf4b71cc3de588ac122eeb8fa878ab464', '1836438502ddccabf165f81210770f1722442376bb2fb75956d5b726f1efc3cb', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.11.contigs.fa_assembly.RAST']


In [ ]:
len(core_data_comp)

In [116]:

#_to_json(core_data_comp['Salt_Pond_MetaG_R2_C_H2O_MG_DASTool_bins_concoct_out.60.contigs__.RAST'])

In [125]:
def _to_json(d):
    ret = {
        'core': d['core'],
        'cluster_to_features': {k: list(v) for k, v in d['cluster_to_features'].items()},
        'feature_id_to_genome': d['feature_id_to_genome']
    }
    return ret
with open('/scratch/fliu/data/cliff/core_data_comp_70.json', 'w') as fh:
    fh.write(json.dumps({k: _to_json(v) for k, v in core_data_comp.items()}))

In [ ]:
core_data

In [91]:
len(core_data['core'])

61

In [26]:
all_genomes = set()
for k in clusters:
    for _k in clusters[k]:
        all_genomes.add(feature_to_genome[_k])

In [28]:
len(all_genomes)

21684

In [47]:
import logging
def load_mmseq_results4(clusters, feature_to_genome, genome_ids, name):
    recs = clusters
    fill = len(str(len(set(clusters.values()))))

    rep2clust = {k: name + str(i).zfill(fill) for i, k in enumerate(set(recs.values()))}
    gene_clusters2rep = {v: k for k, v in rep2clust.items()}

    logging.debug(f'For {len(recs)} CDSes we got {len(gene_clusters2rep)}  gene-clusters')

    recs = {k: rep2clust[v] for k, v in recs.items()}
    genome2gene_clusters = {k: set() for k in genome_ids}

    for k, v in recs.items():
        g = feature_to_genome.get(k)
        if g:
            genome2gene_clusters[g].add(v)
        else:
            logging.warning(f'not found {k}')

    return genome2gene_clusters, recs, gene_clusters2rep

In [ ]:
s_clusters

In [31]:
from modelseedpy_ext.mmseqs.motu import FastMOTU, load_mmseq_results
genome2gene_clusters, recs, gene_clusters2rep = load_mmseq_results4(s_clusters, feature_to_genome, all_genomes, 'c')

TypeError: unhashable type: 'list'

In [ ]:
def get_recs(filter_features):
    recs = {}
    for o in anme_all_cc:
        cc_filter = set(o & filter_features)
        if len(cc_filter) > 0:
            rep = cc_filter.pop()
            recs[rep] = rep
            for i in cc_filter:
                recs[i] = rep
    return recs
tt = {
            'genome2gene_clusterss' : genome2gene_clusters2, 
            'aa2gene_clusters' : recs2, 
            'gene_clusters2rep' : gene_clusters2rep2}

In [32]:
import networkx as nx


def read_clusters_dict(filename, sep='\t'):
    clusters = {}
    with open(filename, 'r') as fh:
        line = fh.readline()
        while line:
            if line:
                rep, member = line.strip().split(sep)
                if rep not in clusters:
                    clusters[rep] = set()
                clusters[rep].add(member)
            line = fh.readline()
        return clusters


def read_clusters(filename, sep='\t'):
    g = nx.Graph()
    with open(filename, 'r') as fh:
        line = fh.readline()
        while line:
            if line:
                rep, member = line.strip().split(sep)
                g.add_edge(rep, member)
            line = fh.readline()
        return g

In [33]:
super_graph = read_clusters('/scratch/fliu/data/cliff/ani_mmseqs/output/mmseqs_cluster.tsv')

In [34]:
all_cc = {frozenset(o) for o in nx.connected_components(super_graph)}

In [35]:
len(all_cc)

4234844

In [38]:
filter_features = set()
for a, b in s_clusters.items():
    filter_features |= set(b)
recs = get_recs(filter_features, all_cc)

In [64]:
genome2gene_clusters2, recs2, gene_clusters2rep2 = load_mmseq_results4(recs, feature_to_genome, f, 'cc')
tt = {
            'genome2gene_clusterss' : genome2gene_clusters2, 
            'aa2gene_clusters' : recs2, 
            'gene_clusters2rep' : gene_clusters2rep2}

In [65]:
fast_motu = FastMOTU(tt)
fast_motu.run()
print(len(fast_motu), fast_motu.mean_comp)

iteration 1 :  1263 sum_abs_LLHR: 60870.59199066482


10 100.0


iteration 2 :  1263 sum_abs_LLHR: 60870.59199066482

Your default-run for 10 genomes (with mean initial completeness 99.90) resulted
in a core of 1263 traits with a total sum of loglikelihood-ratios 60870.59 and a corrected 
mean completness of 99.90, resulting to a estimated mean traits per genome count of 2347.95



In [66]:
m_out = get_stats(fast_motu, 'hi')

In [74]:
cluster_to_features = {}
for f_id, c_id in fast_motu.aa2gene_clusters.items():
    if c_id not in cluster_to_features:
        cluster_to_features[c_id] = set()
    cluster_to_features[c_id].add(f_id)
feature_id_to_genome = {}
for feature_ids in cluster_to_features.values():
    for feature_id in feature_ids:
        genome_id = feature_to_genome.get(feature_id)
        if genome_id:
            feature_id_to_genome[feature_id] = genome_id
        else:
            print('!!')
core_data =  {
    'core': m_out['hi']['core'],
    'cluster_to_features': cluster_to_features,
    'feature_id_to_genome': feature_id_to_genome
}

In [76]:
core_data

{'core': ['cc3346',
  'cc0996',
  'cc3121',
  'cc1161',
  'cc0587',
  'cc4387',
  'cc0056',
  'cc1330',
  'cc3933',
  'cc0774',
  'cc3188',
  'cc3044',
  'cc0799',
  'cc0167',
  'cc1332',
  'cc3440',
  'cc2708',
  'cc0888',
  'cc4098',
  'cc0684',
  'cc4050',
  'cc1967',
  'cc1936',
  'cc2838',
  'cc2357',
  'cc4244',
  'cc1160',
  'cc1480',
  'cc1494',
  'cc2634',
  'cc2538',
  'cc3877',
  'cc2616',
  'cc3046',
  'cc1370',
  'cc3654',
  'cc0450',
  'cc2264',
  'cc2855',
  'cc0453',
  'cc4271',
  'cc4477',
  'cc3738',
  'cc1802',
  'cc2397',
  'cc3070',
  'cc3231',
  'cc0638',
  'cc4300',
  'cc4052',
  'cc1787',
  'cc2143',
  'cc3726',
  'cc2433',
  'cc2875',
  'cc2624',
  'cc0942',
  'cc0231',
  'cc2725',
  'cc2661',
  'cc0988',
  'cc3255',
  'cc3394',
  'cc2790',
  'cc1884',
  'cc0250',
  'cc1244',
  'cc1181',
  'cc0159',
  'cc3399',
  'cc2153',
  'cc2715',
  'cc4160',
  'cc0100',
  'cc0555',
  'cc2317',
  'cc4337',
  'cc3707',
  'cc4094',
  'cc4177',
  'cc2491',
  'cc1217',
  'cc129

In [84]:
from mOTUlizer.classes.MetaBin import MetaBin
from math import log10
import sys

mean = lambda x : sum(x)/len(x)

class FastMOTU:
    
    def __init__(self, tt, name='default'):
        self.gene_clusters_dict = tt['genome2gene_clusterss']
        self.aa2gene_clusters = tt['aa2gene_clusters']
        self.quiet = False
        self.name = name
        
    def __getitem__(self, i):
        return self.members[i]
        
    def __len__(self):
        return len(self.members)
    
    def run(self, genome_completion_dict = None, max_it=20):
        if genome_completion_dict is None:
            genome_completion_dict = {k: 100 for k in self.gene_clusters_dict.keys()}
        self.members = [MetaBin(bin_name, 
                                gene_clusterss = self.gene_clusters_dict[bin_name], 
                                faas = None, #self.faas.get(bin_name), 
                                fnas = None, 
                                complet = genome_completion_dict.get(bin_name)) for bin_name in self.gene_clusters_dict.keys()]
        
        self.gene_clustersCounts = {c : 0 for c in set.union(set([gene_clusters for mag in self.members for gene_clusters in mag.gene_clusterss]))}
        for mag in self.members:
            for gene_clusters in mag.gene_clusterss:
                self.gene_clustersCounts[gene_clusters] += 1
        self.mean_comp = mean(genome_completion_dict.values())
        """
        for gene_clusters, counts in self.gene_clustersCounts.items():
            if counts > 30:
                print(gene_clusters, counts)
                print(100*counts, len(self))
                print(100*counts/len(self) > self.mean_comp)
        """
        self.core = {gene_clusters for gene_clusters, counts in self.gene_clustersCounts.items() if (100*counts/len(self)) > self.mean_comp}
        self.likelies = self.__core_likelyhood(max_it = max_it)
        
    def __core_prob(self, gene_clusters, complet = "checkm"):
        comp = lambda mag : (mag.checkm_complet if complet =="checkm" else mag.new_completness)/100
        presence = [log10(comp(mag)) for mag in self if gene_clusters in mag.gene_clusterss]
        abscence = [log10(1 - comp(mag)) for mag in self if gene_clusters not in mag.gene_clusterss]
        return sum(presence + abscence)

    def __pange_prob(self, gene_clusters, core_size, complet = "checkm"):
#        pool_size = sum(self.gene_clustersCounts.values())
        pool_size = sum([c for k,c in  self.gene_clustersCounts.items()])
        comp = lambda mag : (mag.checkm_complet if complet =="checkm" else mag.new_completness)/100
        #presence = [1 - (1-self.gene_clustersCounts[gene_clusters]/pool_size)**(len(mag.gene_clusterss)-(core_size*comp(mag))) for mag in self if gene_clusters in mag.gene_clusterss]
        #abscence = [ (1-self.gene_clustersCounts[gene_clusters]/pool_size)**(len(mag.gene_clusterss)-(core_size*comp(mag))) for mag in self if gene_clusters not in mag.gene_clusterss]

#        mag_prob = {mag : ( 1-1/pool_size )**len(mag.gene_clusterss) for mag in self}
#        mag_prob = {mag : ( 1-1/pool_size )**(len(mag.gene_clusterss)-(core_size*comp(mag))) for mag in self}
#        mag_prob = {mag : ( 1-self.gene_clustersCounts[gene_clusters]/pool_size )**(len(mag.gene_clusterss)-(core_size*comp(mag))) for mag in self}

#        mag_prob = {mag : ( 1-self.gene_clustersCounts[gene_clusters]/pool_size )**(len(mag.gene_clusterss)-(core_size*comp(mag))) for mag in self}
        mag_prob = {mag : ( 1-self.gene_clustersCounts[gene_clusters]/pool_size)**len(mag.gene_clusterss) for mag in self}

        presence = [ log10(1 -   mag_prob[mag]) if mag_prob[mag] < 1 else MIN_PROB                for mag in self if gene_clusters in mag.gene_clusterss]
        abscence = [ log10(      mag_prob[mag]) if mag_prob[mag] > 0 else log10(1-(10**MIN_PROB)) for mag in self if gene_clusters not in mag.gene_clusterss]

        #abscence = [ 1-self.gene_clustersCounts[gene_clusters]/len(self)*comp(mag) for mag in self if gene_clusters not in mag.gene_clusterss]
        #presence = [ self.gene_clustersCounts[gene_clusters]/len(self)*comp(mag) for mag in self if gene_clusters not in mag.gene_clusterss]

        return sum(presence + abscence)

    def __core_likely(self, gene_clusters, complet = "checkm", core_size = 0):
        pange_prob = self.__pange_prob(gene_clusters, core_size, complet)
        return self.__core_prob(gene_clusters, complet) - pange_prob
        
    def __core_likelyhood(self, max_it = 20, likeli_cutof = 0 ):
        likelies = {gene_clusters : self.__core_likely(gene_clusters) for gene_clusters in self.gene_clustersCounts}
        self.core = set([c for c, v in likelies.items() if v > likeli_cutof])
        core_len = len(self.core)
        i = 1
        if not self.quiet:
            print("iteration 1 : ", core_len, "sum_abs_LLHR:" , sum([l if l > 0 else -l for l in likelies.values()]), file = sys.stderr)
        for mag in self:
            if len(self.core) > 0:
                mag.new_completness = 100*len(mag.gene_clusterss.intersection(self.core))/len(self.core)
            else :
                mag.new_completness = 0
            mag.new_completness = mag.new_completness if mag.new_completness < 99.9 else 99.9
            mag.new_completness = mag.new_completness if mag.new_completness > 0 else 0.01
        for i in range(2,max_it):
            likelies = {gene_clusters : self.__core_likely(gene_clusters, complet = "new", core_size = core_len) for gene_clusters in self.gene_clustersCounts}
            old_core = self.core
            self.core = set([c for c, v in likelies.items() if v > likeli_cutof])
            new_core_len = len(self.core)
            for mag in self:
                if len(self.core) > 0:
                    mag.new_completness = 100*len(mag.gene_clusterss.intersection(self.core))/len(self.core)
                else :
                    mag.new_completness = 0
                mag.new_completness = mag.new_completness if mag.new_completness < 99.9 else 99.9
                mag.new_completness = mag.new_completness if mag.new_completness > 0 else 0.01
            if not self.quiet:
                print("iteration",i, ": ", new_core_len, "sum_abs_LLHR:" , sum([l if l > 0 else -l for l in likelies.values()]), file = sys.stderr)
            if self.core == old_core:
                break
            else :
                core_len = new_core_len

        if not self.quiet:
            pp =  "\nYour {name}-run for {nb_mags} genomes (with mean initial completeness {mean_start:.2f}) resulted\n"
            pp += "in a core of {core_len} traits with a total sum of loglikelihood-ratios {llhr:.2f} and a corrected \n"
            pp += "mean completness of {mean_new:.2f}, resulting to a estimated mean traits per genome count of {trait_count:.2f}\n"
            pp = pp.format(name = self.name, nb_mags = len(self), core_len = core_len, mean_start = mean([b.checkm_complet for b in self]),
                        mean_new =  mean([b.new_completness for b in self]), llhr =  sum([l if l > 0 else -l for l in likelies.values()]),
                        trait_count = mean([100*len(b.gene_clusterss)/b.new_completness for b in self]))
            print(pp, file = sys.stderr)
        self.iterations = i -1
        return likelies
    
def get_stats(self, name):
    out = {}

    out[name] = { "nb_genomes" : len(self),
    "core" : list(self.core) if self.core else None,
    "aux_genome" : [k for k,v in self.gene_clustersCounts.items() if k not in self.core] if self.core else None ,
    "singleton_gene_clusterss" : [k for k,v in self.gene_clustersCounts.items() if k not in self.core if v == 1] if self.core else None,
    "gene_clusterss" : None if self.gene_clusters_dict is None else {'genome' : {k : list(v) for k,v in self.gene_clusters_dict.items()}, 'aa' : self.aa2gene_clusters} if self.aa2gene_clusters else ({k : list(v) for k,v  in self.gene_clusters_dict.items()} if self.gene_clusters_dict else None),
    "mean_ANI" : self.get_mean_ani() if (hasattr(self, 'fastani_dict') or all([hasattr(g, "genome") for g in self])) else None,
    "ANIs" : [[k[0], k[1], v] for k, v in self.fastani_matrix().items()] if (hasattr(self, 'fastani_dict')  or all([hasattr(g, "genome") for g in self])) else None,
    "genomes" : [v.get_data() for v in self],
    "likelies" : self.likelies
    }
    return out

def get_recs(filter_features, cc):
    recs = {}
    for o in cc:
        cc_filter = set(o & filter_features)
        if len(cc_filter) > 0:
            rep = cc_filter.pop()
            recs[rep] = rep
            for i in cc_filter:
                recs[i] = rep
    return recs